# Resume Training — Transformer model from iter 22,000

This notebook is used to continue training from `best_model.pt` (saved at iter 21,000, val loss 3.3716).
Uses the same `combined_tokens.bin` (356M tokens) already on disk.
Remaining: ~28,000 iterations with LR starting at 6.1e-5 (cosine decay).

#### Author: Parth Jade BT2024034

In [ ]:
import os
import re
import math
import random
import gc
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2TokenizerFast

/home/usera19/miniconda3/envs/parthj/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
CUDA version: 12.1
VRAM: 16.7 GB


In [ ]:
class DataManager:
    def __init__(self, data=None):
        self.data=data
        self.train_data=None
        self.val_data = None

    def load_from_bin(self, path, total_tokens):
        print(f"[INFO] Loading tokens from {path}...")
        arr = np.fromfile(path, dtype=np.int32)
        assert len(arr) == total_tokens, f"Expected {total_tokens}, got {len(arr)}"
        self.data = torch.from_numpy(arr).to(torch.long)
        print(f"[INFO] Loaded {len(self.data):,} tokens")
        print(f"[INFO] Tensor size: {self.data.element_size() * len(self.data) / 1e9:.2f} GB")
        return self

    def split(self, ratio=0.9):
        n = int(ratio * len(self.data))
        self.train_data = self.data[:n].contiguous()
        self.val_data   = self.data[n:].contiguous()
        del self.data
        self.data = None
        gc.collect()

        print(f"Train tokens : {len(self.train_data):,}")
        print(f"Val tokens   : {len(self.val_data):,}")
        print(f"Train size   : {self.train_data.element_size() * len(self.train_data) / 1e9:.2f} GB")
        print(f"Val size     : {self.val_data.element_size() * len(self.val_data) / 1e9:.2f} GB")
        return self

    def get_data(self):
        return self.train_data, self.val_data

In [ ]:
class Config:
    """
    config contains parameters which is used to continue training from iter 22,000.
    Architecture is identical. Training hyperparams adjusted:
      - max_iters = 28000 (remaining from 50K total)
      - learning_rate = 6.1e-5 (where LR was during the iter 22K)
    """
    def __init__(self):
        self.batch_size    = 16
        self.block_size    = 256
        self.max_iters     = 28000    #28000 iterations remaaining to train remaining 
        self.learning_rate = 6.1e-5   #LearningRate was ~0.000061 at iter 22000
        self.n_embd        = 512
        self.n_head        = 8
        self.n_layer       = 8
        self.grad_accum    = 8
        self.dropout       =0.1
        self.eval_interval = 500
        self.eval_iters    = 100
        self.warmup_steps  = 100     
        self.patience      = 10
        self.max_grad_norm = 1.0
        self.device        = "cuda" if torch.cuda.is_available() else "cpu"

    def summary(self):
        print("\n===== CONFIG (RESUME) =====")
        for k, v in vars(self).items():
            print(f"{k:15}: {v}")
        return self

In [ ]:
class BatchLoader:
    def __init__(self, train_data, val_data, block_size, batch_size, device):
        self.train_data = train_data
        self.val_data   = val_data
        self.block_size= block_size
        self.batch_size = batch_size
        self.device     = device

    def get_batch(self, split):
        src = self.train_data if split == "train" else self.val_data
        ix = torch.randint(0, len(src) - self.block_size - 1, (self.batch_size,))

        x = torch.stack([src[i : i + self.block_size] for i in ix])
        y = torch.stack([src[i + 1 : i + 1 + self.block_size] for i in ix])

        if self.device == "cuda":
            x = x.pin_memory().to(self.device, non_blocking=True)
            y = y.pin_memory().to(self.device, non_blocking=True)
        else:
            x = x.to(self.device)
            y = y.to(self.device)
        return x, y

In [7]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size)).bool()
        )
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(~self.tril[:T, :T], float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.attn_dropout(wei)
        v = self.value(x)
        return wei @ v

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size= n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)
        ])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.resid_dropout(self.proj(out))

In [9]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa  = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff  = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x +self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout, device):
        super().__init__()
        self.device = device
        self.block_size= block_size

        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])

        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding_table.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)
        pos = self.position_embedding_table(torch.arange(T, device=self.device))
        x   =self.blocks(tok + pos)
        x   = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, -1),
                targets.view(B * T)
            )
        return logits, loss
    
    def generate(
        self,
        idx,
        max_new_tokens,
        temperature=0.9,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3
    ):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
    
            logits = logits[:, -1, :]
    
            if repetition_penalty != 1.0:
                for b in range(idx.size(0)):
                    seen_tokens = set(idx[b].tolist())
                    for token_id in seen_tokens:
                        logits[b, token_id] /= repetition_penalty
    
            if no_repeat_ngram_size > 0 and idx.size(1) >= no_repeat_ngram_size:
                for b in range(idx.size(0)):
                    tokens = idx[b].tolist()
                    n = no_repeat_ngram_size
    
                    ngrams = set(
                        tuple(tokens[i:i+n])
                        for i in range(len(tokens) - n + 1)
                    )
    
                    prefix = tuple(tokens[-(n-1):])
    
                    for token in range(logits.size(-1)):
                        if prefix + (token,) in ngrams:
                            logits[b, token] = -float("inf")
    
            logits = logits / temperature
    
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1, None]] = -float("inf")
    
            probs = F.softmax(logits, dim=-1)
    
            if top_p < 1.0:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cumulative = torch.cumsum(sorted_probs, dim=-1)
    
                mask = cumulative > top_p
                mask[..., 1:] = mask[..., :-1].clone()
                mask[..., 0] = False
    
                sorted_probs[mask] = 0
                probs = torch.zeros_like(probs).scatter_(1, sorted_idx, sorted_probs)
                probs = probs / probs.sum(dim=-1, keepdim=True)
    
            # Sample
            idx_next = torch.multinomial(probs, num_samples=1)
    
            idx = torch.cat((idx, idx_next), dim=1)
    
        return idx

In [ ]:
class Trainer:
    def __init__(self, model, config, batch_loader):
        self.model = model
        self.config = config
        self.batch_loader = batch_loader

        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate
        )

        self.scaler = torch.amp.GradScaler("cuda", enabled=(config.device == "cuda"))

        self.best_val_loss = float("inf")
        self.no_improve_steps = 0

    def get_lr(self, step):
        if step < self.config.warmup_steps:
            return self.config.learning_rate * step / self.config.warmup_steps
        progress = (step - self.config.warmup_steps) / (self.config.max_iters - self.config.warmup_steps)
        return 0.5 * self.config.learning_rate * (1 + math.cos(math.pi * progress))

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        losses = {}

        for split in ["train", "val"]:
            split_losses = torch.zeros(self.config.eval_iters)

            for k in range(self.config.eval_iters):
                x, y = self.batch_loader.get_batch(split)

                with torch.amp.autocast("cuda", enabled=(self.config.device == "cuda")):
                    _, loss = self.model(x, y)

                split_losses[k] = loss.item()
                del x, y, loss

            losses[split] = split_losses.mean().item()

        torch.cuda.empty_cache()
        self.model.train()
        return losses

    def train(self):
        print("\n===== TRAINING RESUMED (from iter ~22000, 28000 iters remaining) =====")

        for iter in range(self.config.max_iters):
            lr = self.get_lr(iter)
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr

            total_loss = 0.0

            for _ in range(self.config.grad_accum):
                x, y = self.batch_loader.get_batch("train")

                with torch.amp.autocast("cuda", enabled=(self.config.device == "cuda")):
                    _, loss = self.model(x, y)
                    loss = loss / self.config.grad_accum

                self.scaler.scale(loss).backward()
                total_loss += loss.item()
                del x, y, loss

            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)

            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad(set_to_none=True)

            if iter % self.config.eval_interval == 0:
                losses = self.evaluate()

                orig_iter = iter + 22000
                print(f"iter {orig_iter:6d} | "
                      f"train {losses['train']:.4f} | "
                      f"val {losses['val']:.4f} | "
                      f"lr {lr:.6f}")

                if losses["val"] < self.best_val_loss:
                    self.best_val_loss = losses["val"]
                    self.no_improve_steps = 0
                    torch.save(self.model.state_dict(), "best_model.pt")
                    print("  -> best model saved")
                else:
                    self.no_improve_steps += 1

                if self.no_improve_steps >= self.config.patience:
                    print("Early stopping triggered")
                    break

        print("===== TRAINING COMPLETE =====")

## Load existing tokenized data
Uses the `combined_tokens.bin` already on disk (356M tokens).
So no re-tokenization needed.

In [ ]:
total_tokens = 356_442_343

data_manager = DataManager()
data_manager.load_from_bin("combined_tokens.bin", total_tokens)
data_manager.split()
train_data, val_data = data_manager.get_data()
del data_manager
gc.collect()

vocab_size = 50257
print(f"\n[INFO] vocab_size = {vocab_size}")

[INFO] Loading tokens from combined_tokens.bin...
[INFO] Loaded 356,442,343 tokens
[INFO] Tensor size: 2.85 GB
Train tokens : 320,798,108
Val tokens   : 35,644,235
Train size   : 2.57 GB
Val size     : 0.29 GB

[INFO] vocab_size = 50257


## Config and BatchLoader

In [14]:
config = Config().summary()


===== CONFIG (RESUME) =====
batch_size     : 16
block_size     : 256
max_iters      : 28000
learning_rate  : 6.1e-05
n_embd         : 512
n_head         : 8
n_layer        : 8
grad_accum     : 8
dropout        : 0.1
eval_interval  : 500
eval_iters     : 100
warmup_steps   : 100
patience       : 10
max_grad_norm  : 1.0
device         : cuda


In [15]:
batch_loader = BatchLoader(
    train_data,
    val_data,
    config.block_size,
    config.batch_size,
    config.device
)

## Build model and load checkpoint
Loads `best_model.pt` saved at iter 21,000 (val loss 3.3716).

In [ ]:
model = GPT(
    vocab_size,
    config.n_embd,
    config.n_head,
    config.n_layer,
    config.block_size,
    config.dropout,
    config.device
).to(config.device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {total_params:,}")

Model parameters : 51,070,464


In [ ]:
#load checkpoint from interrupted training
checkpoint_path = "best_model.pt"

print(f"[INFO] Loading checkpoint from {checkpoint_path}...")
raw_state = torch.load(checkpoint_path, map_location=config.device)

if isinstance(raw_state, dict) and "model_state_dict" in raw_state:
    raw_state = raw_state["model_state_dict"]

cleaned_state = {}
for k, v in raw_state.items():
    new_key = k.replace("_orig_mod.", "")
    cleaned_state[new_key] = v

model.load_state_dict(cleaned_state, strict=True)
print(f"[INFO] Checkpoint loaded successfully!")
print(f"[INFO] All {len(cleaned_state)} parameter tensors matched.")
print(f"[INFO] Resuming from val loss ~3.3716 (iter 21000)")

del raw_state, cleaned_state
gc.collect()
torch.cuda.empty_cache()

model = torch.compile(model)
print("[INFO] Model compiled and ready to resume training.")

[INFO] Loading checkpoint from best_model.pt...
[INFO] Checkpoint loaded successfully!
[INFO] All 341 parameter tensors matched.
[INFO] Resuming from val loss ~3.3716 (iter 21000)


/tmp/ipykernel_299916/2745870158.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  raw_state = torch.load(checkpoint_path, map_location=config.device)


[INFO] Model compiled and ready to resume training.


## Resume training
Continues for 28,000 more iterations (iters 22,000 → 50,000).
Iter numbers in the log are adjusted to show the original count.

In [18]:
trainer = Trainer(model, config, batch_loader)
trainer.train()


===== TRAINING RESUMED (from iter ~22000, 28000 iters remaining) =====
iter  22000 | train 3.4603 | val 3.3248 | lr 0.000000
  -> best model saved
iter  22500 | train 3.5012 | val 3.3396 | lr 0.000061
iter  23000 | train 3.4857 | val 3.3138 | lr 0.000061
  -> best model saved
iter  23500 | train 3.4615 | val 3.3206 | lr 0.000061
iter  24000 | train 3.4600 | val 3.3361 | lr 0.000060
iter  24500 | train 3.4519 | val 3.3259 | lr 0.000060
iter  25000 | train 3.4636 | val 3.3314 | lr 0.000059
iter  25500 | train 3.4467 | val 3.3340 | lr 0.000059
iter  26000 | train 3.4539 | val 3.3131 | lr 0.000058
  -> best model saved
iter  26500 | train 3.4681 | val 3.3113 | lr 0.000057
  -> best model saved
iter  27000 | train 3.4480 | val 3.3240 | lr 0.000056
iter  27500 | train 3.4547 | val 3.3018 | lr 0.000056
  -> best model saved
iter  28000 | train 3.4412 | val 3.3223 | lr 0.000055
iter  28500 | train 3.4456 | val 3.3252 | lr 0.000053
iter  29000 | train 3.4183 | val 3.2683 | lr 0.000052
  -> bes

## Step 5: Save final checkpoint

In [19]:
torch.save({
    "model_state_dict": model.state_dict(),
    "config": vars(config),
    "vocab_size": vocab_size,
    "total_iters": 50000,
    "note": "Resumed from iter 22000, trained to 50000"
}, "final_checkpoint.pt")
print("Final checkpoint saved.")

Final checkpoint saved.


#### Quick generation test

In [20]:
tok = GPT2TokenizerFast.from_pretrained("gpt2")

prompts = [
    "The old library stood at the edge of town",
    "She opened the door and",
    "In the year 2050, humanity had finally",
]

model.eval()
with torch.no_grad():
    for prompt in prompts:
        idx = torch.tensor([tok.encode(prompt)], device=config.device)
        out = model.generate(idx, max_new_tokens=150, temperature=0.8, top_k=40, top_p=0.85)
        print(f"\n{'='*60}")
        print(f"PROMPT: {prompt}")
        print(f"{'='*60}")
        print(tok.decode(out[0].tolist()))


PROMPT: The old library stood at the edge of town
The old library stood at the edge of town , and there were two more buildings in the upper level . The library had a library , a library , a library , and a library .
The library was the only library in the world to have a library , and a library . There were books of art and other religious themes and traditions . The library was housed in the library 's library , and the library was the only library in the world to receive the library . The library was used to decorate the library and libraries . The library also included the library 's library , which was the first library in the world to have an library . The library was the library 's library , which was a library , and the library itself was also used to research and study books , such as the

PROMPT: She opened the door and
She opened the door and found that " the whole thing was like a great piece of glass " . She went inside , and sat in a chair in the corner .
After the First